## SAX, Hierarchical clustering and volatility prediction

This notebook keeps the data-specific loading, filtering, interpolation, denoising, detrending, and robust normalization logic unchanged in spirit, while replacing the analysis layer with:

1. official `tslearn` SAX representations,
2. SAX MINDIST distance on symbolic strings,
3. agglomerative hierarchical clustering with complete linkage,
4. subsample-based clustering stability by `k`,
5. explainable cluster-level Google Trends indices, and
6. compact GARCH-related market prediction diagnostics.

Outputs are prioritized as CSV files. Only a small number of diagnostic figures are saved.

## 0. Configuration

Edit paths and headline modeling choices here. The rest of the notebook is designed to run top-to-bottom.

In [86]:
from __future__ import annotations

import json
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.signal import savgol_filter
from scipy.spatial.distance import squareform
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from tslearn.piecewise import SymbolicAggregateApproximation
from arch import arch_model

# -----------------------------------------------------------------------------
# PATHS -- keep folder structure consistent with the current project
# -----------------------------------------------------------------------------
DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_09")

SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
DJ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\DOWJONES_data.csv")
NASDAQ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\NASDAQ100_data.csv")
RUSSELL_PATH = Path(r"C:\Python\CSUREMM\shock_detection\RUSSELL2000_data.csv")

STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed02*.csv"

START_DATE = "2022-01-01"
END_DATE = "2026-05-31"

# -----------------------------------------------------------------------------
# Data-specific filtering and preprocessing -- preserved from the current setup
# -----------------------------------------------------------------------------
MAX_ZERO_SHARE = 0.50
MAX_MISSING_SHARE = 0.02
MAX_INTERPOLATE_GAP = 7

DENOISE_WINDOW = 9
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91

# -----------------------------------------------------------------------------
# SAX and hierarchical clustering
# -----------------------------------------------------------------------------
MINDIST_CHUNK_SIZE = 256
K_RANGE = range(2, 13)
# FINAL_K = 3
RANDOM_STATE = 42
N_SEGMENTS = 96
ALPHABET_SIZE = 9
FINAL_K = 6

# Stability: each iteration draws two overlapping subsamples and compares labels
STABILITY_SUBSAMPLES = 80
STABILITY_FRACTION = 0.80

# Cluster-index construction
CORE_STABILITY_QUANTILE = 0.50
MIN_CORE_TERMS = 10

# Prediction
TARGET_HORIZONS = [1, 5]
MIN_TRAIN = 500
RIDGE_ALPHAS = np.logspace(-4, 4, 25)

SUBDIRS = [
    "00_provenance",
    "01_preprocessed_panel",
    "02_sax",
    "03_hierarchical_clustering",
    "04_stability",
    "05_cluster_indices",
    "06_prediction",
    "07_validity_checks",
]

for sub in SUBDIRS:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("arch package available:", HAS_ARCH)

Output directory: C:\Python\CSUREMM\output\sax_tests_july_09
arch package available: True


## 1. Load, filter, denoise, detrend, and normalize

This section preserves the project-specific data logic because it is tied to the Google Trends extraction and stitching process.

In [14]:
def clean_term_from_folder(folder: Path) -> str:
    return folder.name.replace("_", " ").strip()


def find_value_column(df: pd.DataFrame) -> str:
    date_like = {"time", "date", "week"}
    candidates = [c for c in df.columns if c.lower() not in date_like]

    if not candidates:
        raise ValueError("No value column found.")

    numeric = [
        c for c in candidates
        if pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors="coerce"))
    ]

    return numeric[0] if numeric else candidates[0]


def max_consecutive_missing(s: pd.Series) -> int:
    missing = s.isna()

    if not missing.any():
        return 0

    groups = (missing != missing.shift()).cumsum()

    return int(
        missing
        .groupby(groups)
        .sum()
        .max()
    )


def load_one_series(folder: Path) -> pd.Series:
    stitched_dir = folder / STITCHED_SUBDIR
    files = sorted(stitched_dir.glob(STITCHED_GLOB)) if stitched_dir.exists() else []

    if not files:
        raise FileNotFoundError("missing stitched file")

    path = files[-1]
    df = pd.read_csv(path)

    date_col = next(
        (c for c in df.columns if c.lower() in {"time", "date", "week"}),
        df.columns[0],
    )

    value_col = find_value_column(df)

    dates = pd.to_datetime(df[date_col], errors="coerce")
    values = pd.to_numeric(df[value_col], errors="coerce")

    s = pd.Series(
        values.values,
        index=dates,
        name=clean_term_from_folder(folder),
    )

    s = s[~s.index.isna()]
    s = s[~s.index.duplicated(keep="last")]
    s = s.sort_index()

    s = s.loc[START_DATE:END_DATE]

    full_index = pd.date_range(
        START_DATE,
        END_DATE,
        freq="D",
    )

    return s.reindex(full_index)


def build_panel(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    series = {}

    folders = sorted([p for p in data_dir.iterdir() if p.is_dir()])

    expected_days = len(
        pd.date_range(
            START_DATE,
            END_DATE,
            freq="D",
        )
    )

    for folder in folders:
        term = clean_term_from_folder(folder)

        try:
            s = load_one_series(folder)

            observed_share = float(s.notna().mean())
            missing_share = float(s.isna().mean())
            zero_share = float((s.dropna() == 0).mean())
            longest_missing_gap = max_consecutive_missing(s)
            n_unique = int(s.nunique(dropna=True))

            reason = "kept"

            if zero_share > MAX_ZERO_SHARE:
                reason = "high_zero_share"
            elif missing_share > MAX_MISSING_SHARE:
                reason = "high_missing_share"
            elif longest_missing_gap > MAX_INTERPOLATE_GAP:
                reason = "long_missing_gap"
            elif n_unique <= 2:
                reason = "near_constant"

            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": int(s.notna().sum()),
                "observed_share": observed_share,
                "missing_share": missing_share,
                "zero_share": zero_share,
                "longest_missing_gap": longest_missing_gap,
                "n_unique": n_unique,
                "drop_reason": reason,
            })

            if reason == "kept":
                series[term] = s

        except Exception as e:
            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": 0,
                "observed_share": 0.0,
                "missing_share": 1.0,
                "zero_share": np.nan,
                "longest_missing_gap": np.nan,
                "n_unique": 0,
                "drop_reason": str(e),
            })

    panel = pd.DataFrame(series).sort_index()
    funnel = pd.DataFrame(rows).sort_values(["drop_reason", "term"])

    return panel, funnel


def interpolate_small_gaps(s: pd.Series) -> pd.Series:
    return (
        s.astype(float)
        .interpolate(
            method="time",
            limit=MAX_INTERPOLATE_GAP,
            limit_direction="both",
        )
    )


def denoise_series(s: pd.Series) -> pd.Series:
    x = s.astype(float)
    valid = x.dropna()

    if len(valid) < DENOISE_WINDOW or DENOISE_WINDOW % 2 == 0:
        return x

    filled = x.interpolate(
        method="time",
        limit_direction="both",
    )

    return pd.Series(
        savgol_filter(
            filled,
            DENOISE_WINDOW,
            DENOISE_POLYORDER,
        ),
        index=x.index,
        name=x.name,
    )


def detrend_series(s: pd.Series) -> pd.Series:
    trend = s.rolling(
        DETREND_WINDOW,
        center=True,
        min_periods=max(14, DETREND_WINDOW // 4),
    ).median()

    return s - trend


def robust_zscore(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    x = s.astype(float)

    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)

        if pd.isna(std) or std == 0:
            return pd.Series(
                np.zeros(len(x)),
                index=x.index,
                name=x.name,
            )

        return (
            (x - x.mean(skipna=True)) / std
        ).rename(x.name)

    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    mad_ratio = (
        mad / wide_spread
        if wide_spread > 0
        else 1.0
    )

    is_degenerate = mad_ratio < mad_ratio_threshold

    if not is_degenerate:
        return (
            0.6745 * (x - med) / mad
        ).rename(x.name)

    mad_floored = (
        max(mad, mad_floor_frac * wide_spread)
        if wide_spread > 0
        else mad
    )

    z = (
        0.6745 * (x - med) / mad_floored
    ).rename(x.name)

    return z.clip(
        lower=-clip_mad_multiple,
        upper=clip_mad_multiple,
    )


def preprocess_panel(
    panel: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:

    stages = {}

    stages["filled"] = panel.apply(
        interpolate_small_gaps,
        axis=0,
    )

    stages["denoised"] = stages["filled"].apply(
        denoise_series,
        axis=0,
    )

    stages["detrended"] = stages["denoised"].apply(
        detrend_series,
        axis=0,
    )

    stages["normalized"] = stages["detrended"].apply(
        robust_zscore,
        axis=0,
    )

    return stages["normalized"], stages


def save_panel_stages(panel_raw: pd.DataFrame, stages: dict[str, pd.DataFrame]) -> None:
    panel_raw.to_csv(OUTPUT_DIR / "01_preprocessed_panel" / "panel_raw_retained.csv")
    for name, df in stages.items():
        df.to_csv(OUTPUT_DIR / "01_preprocessed_panel" / f"panel_{name}.csv")

In [65]:
panel_raw, filtering_funnel = build_panel(DATA_DIR)
filtering_funnel.to_csv(OUTPUT_DIR / "00_provenance" / "filtering_funnel.csv", index=False)

panel_norm, stages = preprocess_panel(panel_raw)
panel_norm = panel_norm.dropna(axis=1, how="all")
panel_norm = panel_norm.loc[:, panel_norm.notna().mean() > 0.95]
stages["normalized"] = panel_norm

save_panel_stages(panel_raw, stages)

summary = pd.DataFrame({
    "n_days": [panel_norm.shape[0]],
    "n_terms_retained": [panel_norm.shape[1]],
    "start_date": [panel_norm.index.min()],
    "end_date": [panel_norm.index.max()],
})
summary.to_csv(OUTPUT_DIR / "00_provenance" / "analysis_sample_summary.csv", index=False)

print(summary.to_string(index=False))
print("\nFiltering funnel:")
print(filtering_funnel["drop_reason"].value_counts(dropna=False).to_string())

 n_days  n_terms_retained start_date   end_date
   1612               847 2022-01-01 2026-05-31

Filtering funnel:
drop_reason
kept                  847
high_zero_share       352
high_missing_share      4


In [66]:
panel_norm.isna().sum().sum() == 0

np.True_

## 2. `tslearn` SAX representation

Each retained term is represented as one SAX string. The notebook uses `tslearn.piecewise.SymbolicAggregateApproximation`. Missing values after preprocessing are linearly filled only for the representation step.

In [71]:
def panel_to_tslearn_array(panel: pd.DataFrame) -> np.ndarray:
    filled = panel.astype(float).interpolate("time").ffill().bfill()
    X = filled.T.to_numpy(dtype=float)
    return X[:, :, None]


def sax_to_strings(sax_codes: np.ndarray, alphabet_size: int) -> pd.Series:
    alphabet = np.array(list("abcdefghijklmnopqrstuvwxyz"[:alphabet_size]))
    codes = sax_codes[:, :, 0].astype(int)
    strings = ["".join(alphabet[row]) for row in codes]
    return pd.Series(strings, index=panel_norm.columns, name="sax_string")

# throwing away extreme values to avoid distortion in SAX representation
panel_sax = panel_norm.clip(-5, 5)

X_ts = panel_to_tslearn_array(panel_sax)
sax = SymbolicAggregateApproximation(
    n_segments=N_SEGMENTS,
    alphabet_size_avg=ALPHABET_SIZE,
    scale=False,
)
sax_codes = sax.fit_transform(X_ts)
sax_strings = sax_to_strings(sax_codes, ALPHABET_SIZE)

sax_df = pd.DataFrame({
    "term": sax_strings.index,
    "sax_string": sax_strings.values,
    "n_segments": N_SEGMENTS,
    "alphabet_size": ALPHABET_SIZE,
})
sax_df.to_csv(OUTPUT_DIR / "02_sax" / "sax_strings.csv", index=False)

symbol_counts = (
    sax_df.assign(symbol=sax_df["sax_string"].map(list))
    .explode("symbol")
    .groupby(["term", "symbol"])
    .size()
    .unstack(fill_value=0)
)
symbol_counts.to_csv(OUTPUT_DIR / "02_sax" / "sax_symbol_counts.csv")

print("SAX strings:", sax_df.shape)
sax_df.head()

SAX strings: (847, 4)


,term,sax_string,n_segments,alphabet_size
0,2020 election map,gfcebedheagifbfcechidfcdbggdggdcfddbiibfeciaed...,96,9
1,2020 election results,fgcgfddhieihaiidabgiiiddeefeieeeedddiiecbhibhf...,96,9
2,2022 election,ddfihcaiifcaaiideaeigfeeeeeeeegdifdeheddghiedc...,96,9
3,2022 election results,eeeigfdiiigebiieedeigeeeeeeeeefefecfgedefeidcd...,96,9
4,2024 election,geefddeeechhhdddecfiicbcfhiahgggieacgifaaiiaaa...,96,9


### Justify clipping value selection - 5 is the right approach

In [75]:
for c in [2,3,4,5]:
    pct = np.mean(np.abs(panel_norm.to_numpy()) > c)
    print(c, pct)

2 0.18607492214530338
3 0.1243272856176082
4 0.09299351674718244
5 0.07572705886488877


## 3. SAX MINDIST and complete-linkage hierarchical clustering

The clustering distance is changed from Dice n-gram dissimilarity to SAX **MINDIST**. This is a minimal methodological update: the notebook still uses the official `tslearn` SAX representation and the same complete-linkage hierarchical clustering workflow, but the distance matrix now uses the lower-bounding SAX distance designed for symbolic aggregate approximation rather than a set/count overlap distance.

For two SAX words, MINDIST sums symbol-to-symbol distances from the Gaussian breakpoint table and rescales by the original series length divided by the number of SAX segments.

In [72]:
def sax_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian breakpoints used by standard SAX."""
    return norm.ppf(np.arange(1, alphabet_size) / alphabet_size)


def sax_symbol_distance_table(alphabet_size: int) -> np.ndarray:
    """
    Pairwise SAX symbol distances used in MINDIST.

    Adjacent symbols have distance 0 because their intervals touch. Non-adjacent
    symbols are separated by the gap between their Gaussian breakpoints.
    """
    bp = sax_breakpoints(alphabet_size)
    table = np.zeros((alphabet_size, alphabet_size), dtype=float)

    for i in range(alphabet_size):
        for j in range(alphabet_size):
            if abs(i - j) <= 1:
                table[i, j] = 0.0
            else:
                lo, hi = sorted((i, j))
                table[i, j] = bp[hi - 1] - bp[lo]
    return table


def sax_mindist_matrix(
    sax_codes: np.ndarray,
    terms: list[str],
    n_timestamps: int,
    n_segments: int,
    alphabet_size: int,
    chunk_size: int = MINDIST_CHUNK_SIZE,
) -> pd.DataFrame:
    """
    Compute SAX MINDIST between all term-level SAX strings.

    Parameters
    ----------
    sax_codes:
        Output from tslearn SymbolicAggregateApproximation, with shape
        (n_terms, n_segments, 1).
    terms:
        Term names in the same order as sax_codes.
    n_timestamps:
        Original equal-length series length before SAX compression.
    n_segments:
        Number of SAX/PAA segments.
    alphabet_size:
        Number of SAX symbols.
    chunk_size:
        Row chunk size to keep memory use stable.

    Returns
    -------
    pd.DataFrame
        Symmetric MINDIST matrix indexed and columned by term.
    """
    codes = np.asarray(sax_codes[:, :, 0], dtype=np.int16)
    n_terms, actual_segments = codes.shape

    if actual_segments != n_segments:
        raise ValueError(f"Expected {n_segments} SAX segments, got {actual_segments}.")
    if len(terms) != n_terms:
        raise ValueError("Number of terms does not match number of SAX code rows.")

    symbol_dist = sax_symbol_distance_table(alphabet_size)
    scale = np.sqrt(n_timestamps / n_segments)
    D = np.zeros((n_terms, n_terms), dtype=np.float32)

    for start in range(0, n_terms, chunk_size):
        stop = min(start + chunk_size, n_terms)
        block = codes[start:stop]

        # Shape: (chunk, n_terms, n_segments)
        per_segment = symbol_dist[block[:, None, :], codes[None, :, :]]
        D[start:stop, :] = scale * np.sqrt((per_segment ** 2).sum(axis=2))

    # Enforce exact symmetry and zero diagonal after float32 chunking.
    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)

    return pd.DataFrame(D, index=terms, columns=terms)


def hierarchical_labels(distance_df: pd.DataFrame, k: int) -> pd.Series:
    condensed = squareform(distance_df.to_numpy(dtype=float), checks=False)
    Z = linkage(condensed, method="complete")
    labels = fcluster(Z, t=k, criterion="maxclust")
    return pd.Series(labels, index=distance_df.index, name="cluster")


mindist = sax_mindist_matrix(
    sax_codes=sax_codes,
    terms=sax_strings.index.to_list(),
    n_timestamps=panel_norm.shape[0],
    n_segments=N_SEGMENTS,
    alphabet_size=ALPHABET_SIZE,
)
mindist.to_csv(OUTPUT_DIR / "03_hierarchical_clustering" / "sax_mindist_matrix.csv")

mindist_summary = pd.DataFrame({
    "metric": ["n_terms", "n_segments", "alphabet_size", "min_offdiag", "median_offdiag", "mean_offdiag", "max_offdiag"],
    "value": [
        mindist.shape[0],
        N_SEGMENTS,
        ALPHABET_SIZE,
        np.nanmin(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmedian(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmean(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmax(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
    ],
})
mindist_summary.to_csv(OUTPUT_DIR / "03_hierarchical_clustering" / "sax_mindist_summary.csv", index=False)

Z = linkage(squareform(mindist.to_numpy(dtype=float), checks=False), method="complete")
labels_final = hierarchical_labels(mindist, FINAL_K)

cluster_assignments = labels_final.rename_axis("term").reset_index()
cluster_assignments.to_csv(
    OUTPUT_DIR / "03_hierarchical_clustering" / "cluster_assignments.csv",
    index=False,
)

cluster_sizes = labels_final.value_counts().sort_index().rename_axis("cluster").reset_index(name="n_terms")
cluster_sizes.to_csv(OUTPUT_DIR / "03_hierarchical_clustering" / "cluster_sizes.csv", index=False)

print(cluster_sizes.to_string(index=False))
mindist_summary

 cluster  n_terms
       1      219
       2       40
       3      207
       4      185
       5      157
       6       39


,metric,value
0,n_terms,847.000000
1,n_segments,96.000000
2,alphabet_size,9.000000
3,min_offdiag,0.000000
4,median_offdiag,31.276871
5,mean_offdiag,30.715071
6,max_offdiag,50.986141


### Some diagnosis on SAX characterization

In [73]:
print("panel_norm shape:", panel_norm.shape)
print("N_SEGMENTS:", N_SEGMENTS)
print("ALPHABET_SIZE:", ALPHABET_SIZE)
print("FINAL_K:", FINAL_K)
print("NaNs:", panel_norm.isna().sum().sum())
print("Constant cols:", (panel_norm.std(axis=0) == 0).sum())

panel_norm shape: (1612, 847)
N_SEGMENTS: 96
ALPHABET_SIZE: 9
FINAL_K: 6
NaNs: 0
Constant cols: 0


In [74]:
codes = np.asarray(sax_codes[:, :, 0], dtype=int)

# 1. How many unique SAX words?
sax_words = ["".join(map(str, row)) for row in codes]
print("unique SAX words:", pd.Series(sax_words).nunique())
print(pd.Series(sax_words).value_counts().head(20))

# 2. Symbol usage
symbol_counts = pd.Series(codes.ravel()).value_counts().sort_index()
print(symbol_counts)

# 3. Per-term number of unique symbols
unique_symbols_per_term = pd.Series(
    [len(set(row)) for row in codes],
    index=sax_strings.index,
    name="n_unique_symbols"
)
print(unique_symbols_per_term.describe())
print(unique_symbols_per_term.value_counts().sort_index())

# 4. Distance distribution
offdiag = mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy().ravel()
offdiag = offdiag[~np.isnan(offdiag)]
print(pd.Series(offdiag).describe())
print(pd.Series(np.round(offdiag, 6)).value_counts().head(20))

unique SAX words: 843
444865388864188443486444444444545425643454832388478100488058300288804575838584456885444885008823    2
644533444277733342588212578076668402685008800081488001660588081467444444444444644853346844335844    2
117820288331088450387644445454687434744374873165786543683778372468444444444874574832344868304848    2
246416577543335576664704766536774432265766627047555267654233676457250565644575533345666564214745    2
652414374068515242783523166366325331881542804388087310441888273466444444444444444645545886216853    1
562653378487088301688833445484444333884217817578286322473688181378444444444544644883135873808881    1
335872088520088340486544444444638534743367843244486442343477222688421444434363366882247786108850    1
424633453377744242488243588166777703685118800286088212370688081467444444444444644863345844236853    1
157308888643336786882803887426884433477867726028663476553334687457250474535564433445765563215645    1
884264464444457686502827844544754444875885084338844226522147

In [76]:
for k in range(2,11):
    labels = hierarchical_labels(mindist, k)

    print(
        k,
        labels.value_counts().values
    )

2 [628 219]
3 [381 247 219]
4 [247 219 196 185]
5 [219 207 196 185  40]
6 [219 207 185 157  40  39]
7 [207 185 181 157  40  39  38]
8 [207 181 157 122  63  40  39  38]
9 [207 181 157 122  63  40  38  23  16]
10 [207 168 157 122  63  40  38  23  16  13]


## 4. Cluster-size selection and subsample stability

For each `k`, the notebook records compact validation metrics and a stability distribution. Stability repeatedly clusters two overlapping subsamples, intersects their shared terms, and computes adjusted Rand similarity on the shared labels.

In [77]:
def labels_from_submatrix(full_distance: pd.DataFrame, terms: list[str], k: int) -> pd.Series:
    sub = full_distance.loc[terms, terms]
    return hierarchical_labels(sub, k)


def stability_for_k(
    full_distance: pd.DataFrame,
    k: int,
    n_subsamples: int,
    fraction: float,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    terms = np.array(full_distance.index)
    n_take = max(k + 2, int(round(len(terms) * fraction)))
    rows = []

    for b in range(n_subsamples):
        s1 = rng.choice(terms, size=n_take, replace=False).tolist()
        s2 = rng.choice(terms, size=n_take, replace=False).tolist()
        common = sorted(set(s1).intersection(s2))

        if len(common) < max(3, k):
            rows.append({"k": k, "iteration": b, "n_common": len(common), "ari": np.nan})
            continue

        l1 = labels_from_submatrix(full_distance, s1, k).loc[common]
        l2 = labels_from_submatrix(full_distance, s2, k).loc[common]
        rows.append({
            "k": k,
            "iteration": b,
            "n_common": len(common),
            "ari": adjusted_rand_score(l1, l2),
        })
    return pd.DataFrame(rows)


def validation_for_k(full_distance: pd.DataFrame, labels: pd.Series, k: int) -> dict:
    D = full_distance.to_numpy()
    labs = labels.loc[full_distance.index].to_numpy()
    sil = silhouette_score(D, labs, metric="precomputed") if len(np.unique(labs)) > 1 else np.nan
    sizes = labels.value_counts()
    return {
        "k": k,
        "silhouette_mindist": sil,
        "min_cluster_size": int(sizes.min()),
        "max_cluster_size": int(sizes.max()),
        "median_cluster_size": float(sizes.median()),
    }

validation_rows = []
stability_tables = []

for k in K_RANGE:
    labels_k = hierarchical_labels(mindist, k)
    validation_rows.append(validation_for_k(mindist, labels_k, k))
    stability_tables.append(
        stability_for_k(
            mindist,
            k=k,
            n_subsamples=STABILITY_SUBSAMPLES,
            fraction=STABILITY_FRACTION,
            seed=RANDOM_STATE + k,
        )
    )

validation_df = pd.DataFrame(validation_rows)
stability_raw = pd.concat(stability_tables, ignore_index=True)
stability_summary = (
    stability_raw
    .groupby("k")
    .agg(
        stability_mean=("ari", "mean"),
        stability_median=("ari", "median"),
        stability_p10=("ari", lambda x: x.quantile(0.10)),
        stability_p90=("ari", lambda x: x.quantile(0.90)),
        mean_common_terms=("n_common", "mean"),
    )
    .reset_index()
)

cluster_selection = validation_df.merge(stability_summary, on="k", how="left")
cluster_selection.to_csv(OUTPUT_DIR / "04_stability" / "cluster_selection_by_k.csv", index=False)
stability_raw.to_csv(OUTPUT_DIR / "04_stability" / "subsample_stability_raw.csv", index=False)

print(cluster_selection.sort_values(["stability_median", "silhouette_mindist"], ascending=False).to_string(index=False))

 k  silhouette_mindist  min_cluster_size  max_cluster_size  median_cluster_size  stability_mean  stability_median  stability_p10  stability_p90  mean_common_terms
12            0.029579                13               168                 39.5        0.231147          0.224393       0.176385       0.284020           542.7750
10            0.043756                13               207                 51.5        0.217397          0.217303       0.160277       0.272721           542.8750
11            0.045629                13               207                 39.0        0.216012          0.213016       0.167471       0.265651           543.5250
 8            0.039559                38               207                 92.5        0.189707          0.191302       0.134212       0.242366           543.6125
 9            0.041682                16               207                 63.0        0.192887          0.184798       0.142202       0.264618           544.0375
 7            0.036644

In [78]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cluster_selection["k"], cluster_selection["stability_median"], marker="o", label="median stability")
ax.plot(cluster_selection["k"], cluster_selection["silhouette_mindist"], marker="o", label="silhouette")
ax.axvline(FINAL_K, linestyle="--", linewidth=1)
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Score")
ax.set_title("Hierarchical cluster selection")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_stability" / "cluster_selection_scores.png", dpi=220)
plt.close(fig)

In [111]:
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt

D = mindist.to_numpy(dtype=float)
condensed = squareform(D, checks=False)
Z_final = linkage(condensed, method="complete")

labels_final = fcluster(Z_final, t=FINAL_K, criterion="maxclust")
labels_final = pd.Series(labels_final, index=mindist.index, name="cluster")

cluster_sizes_final = (
    labels_final
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="n_terms")
)

cluster_sizes_final.to_csv(
    OUTPUT_DIR / "04_stability" / "cluster_sizes_final.csv",
    index=False,
)

plt.figure(figsize=(14, 6))
dendrogram(
    Z_final,
    no_labels=True,
    color_threshold=Z_final[-(FINAL_K - 1), 2],
    above_threshold_color="gray",
)
plt.title(f"Hierarchical clustering dendrogram, k = {FINAL_K}")
plt.xlabel("Search terms")
plt.ylabel("SAX MINDIST")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "04_stability" / f"dendrogram_k{FINAL_K}.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

cluster_sizes_final

,cluster,n_terms
0,1,219
1,2,40
2,3,207
3,4,185
4,5,157
5,6,39


## 5. Term-level stability, cluster summaries, and explainable indices

The final cluster labels are converted into interpretable time-series indices. Three versions are saved:

- equal-weighted cluster index,
- stability-weighted cluster index,
- core-term equal-weighted index using the more stable terms within each cluster.

Indices are built from the normalized panel for predictive modeling and from the denoised level panel for interpretability checks.

In [84]:
# -----------------------------------------------------------------------------
# Cluster indices and interpretable cluster summaries
# Inputs already defined:
#   mindist, labels_final, panel_norm, stages, OUTPUT_DIR
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd

INDEX_DIR = OUTPUT_DIR / "05_cluster_indices"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

TOP_N_CORE_TERMS = 10
CORE_STABILITY_QUANTILE = 0.75
MIN_CORE_TERMS = 10


def term_stability_from_distance(
    full_distance: pd.DataFrame,
    final_labels: pd.Series,
) -> pd.DataFrame:
    rows = []

    for term in final_labels.index:
        cl = final_labels.loc[term]

        same_terms = final_labels.index[
            (final_labels == cl) & (final_labels.index != term)
        ]

        other_terms = final_labels.index[final_labels != cl]

        intra_dist = (
            full_distance.loc[term, same_terms].mean()
            if len(same_terms) > 0
            else np.nan
        )

        between_dist = (
            full_distance.loc[term, other_terms].mean()
            if len(other_terms) > 0
            else np.nan
        )

        margin = (
            between_dist - intra_dist
            if pd.notna(intra_dist) and pd.notna(between_dist)
            else np.nan
        )

        rows.append({
            "term": term,
            "cluster": int(cl),
            "mean_intra_mindist": intra_dist,
            "mean_between_mindist": between_dist,
            "mindist_margin": margin,
        })

    df = pd.DataFrame(rows)

    df["term_stability"] = (
        df.groupby("cluster")["mindist_margin"]
        .rank(pct=True, method="average")
    )

    return df.sort_values(
        ["cluster", "term_stability"],
        ascending=[True, False],
    )


def select_core_terms(
    term_stability: pd.DataFrame,
    top_n: int = TOP_N_CORE_TERMS,
) -> pd.DataFrame:
    rows = []

    for cl, g in term_stability.groupby("cluster"):
        g = g.sort_values(
            ["term_stability", "mindist_margin"],
            ascending=[False, False],
        ).copy()

        q = g["term_stability"].quantile(CORE_STABILITY_QUANTILE)
        core = g[g["term_stability"] >= q].copy()

        min_keep = min(MIN_CORE_TERMS, len(g))
        if len(core) < min_keep:
            core = g.head(min_keep).copy()

        core["core_rank"] = np.arange(1, len(core) + 1)
        rows.append(core)

    return pd.concat(rows, ignore_index=True)


def weighted_average(panel: pd.DataFrame, weights: pd.Series) -> pd.Series:
    cols = [c for c in weights.index if c in panel.columns]

    if len(cols) == 0:
        return pd.Series(np.nan, index=panel.index)

    w = weights.loc[cols].astype(float).fillna(0.0)

    if w.sum() <= 0:
        w = pd.Series(1 / len(cols), index=cols)
    else:
        w = w / w.sum()

    return panel[cols].mul(w, axis=1).sum(axis=1)


def build_cluster_indices(
    panel: pd.DataFrame,
    labels: pd.Series,
    term_stability: pd.DataFrame,
    core_terms: pd.DataFrame,
    suffix: str,
) -> pd.DataFrame:
    out = pd.DataFrame(index=panel.index)
    stab = term_stability.set_index("term")["term_stability"]

    for cl in sorted(labels.unique()):
        members = labels[labels == cl].index.intersection(panel.columns).to_list()
        core = (
            core_terms.loc[core_terms["cluster"] == cl, "term"]
            .pipe(lambda s: s[s.isin(panel.columns)])
            .to_list()
        )

        out[f"cluster_{cl}_equal_{suffix}"] = panel[members].mean(axis=1)

        out[f"cluster_{cl}_stability_weighted_{suffix}"] = weighted_average(
            panel,
            stab.loc[members].fillna(0.0) + 1e-6,
        )

        out[f"cluster_{cl}_core_equal_{suffix}"] = (
            panel[core].mean(axis=1) if len(core) > 0 else np.nan
        )

    return out


# -----------------------------------------------------------------------------
# Run summaries
# -----------------------------------------------------------------------------

term_stability = term_stability_from_distance(mindist, labels_final)
core_terms = select_core_terms(term_stability, top_n=TOP_N_CORE_TERMS)

term_stability.to_csv(INDEX_DIR / "term_stability.csv", index=False)
core_terms.to_csv(INDEX_DIR / "core_terms.csv", index=False)


# -----------------------------------------------------------------------------
# Build indices
# -----------------------------------------------------------------------------

indices_norm = build_cluster_indices(
    panel=panel_norm,
    labels=labels_final,
    term_stability=term_stability,
    core_terms=core_terms,
    suffix="norm",
)

indices_level = build_cluster_indices(
    panel=stages["denoised"].reindex(panel_norm.index),
    labels=labels_final,
    term_stability=term_stability,
    core_terms=core_terms,
    suffix="level",
)

indices_all = indices_norm.join(indices_level)

indices_all.to_csv(INDEX_DIR / "cluster_indices.csv")


# -----------------------------------------------------------------------------
# Cluster-level summary
# -----------------------------------------------------------------------------

cluster_summary = (
    term_stability
    .groupby("cluster")
    .agg(
        n_terms=("term", "size"),
        median_term_stability=("term_stability", "median"),
        mean_intra_mindist=("mean_intra_mindist", "mean"),
        mean_between_mindist=("mean_between_mindist", "mean"),
        mean_mindist_margin=("mindist_margin", "mean"),
    )
    .reset_index()
)

top_terms = (
    core_terms
    .sort_values(["cluster", "core_rank"])
    .groupby("cluster")["term"]
    .apply(lambda x: ", ".join(x.head(TOP_N_CORE_TERMS)))
)

cluster_summary["top_10_core_terms"] = (
    cluster_summary["cluster"]
    .map(top_terms)
)

cluster_summary.to_csv(INDEX_DIR / "cluster_summary.csv", index=False)

print(cluster_summary.to_string(index=False))

 cluster  n_terms  median_term_stability  mean_intra_mindist  mean_between_mindist  mean_mindist_margin                                                                                                                                                               top_10_core_terms
       1      219               0.502283           27.319458             31.670475             4.351016                                                   google translate, 2048, unblocked games, clever, google drive, translate, slope, docs, english to spanish, spanish to english
       2       40               0.512500           24.216293             33.665489             9.449199                                                          nfl, browns, pittsburgh steelers, cleveland browns, cowboys, miami dolphins, bears, dallas cowboys, dolphins, steelers
       3      207               0.502415           26.429350             29.856316             3.426965                                                         

## 6. Market targets and compact prediction workflow

The prediction layer saves one CSV for rolling ridge out-of-sample performance and one CSV for GARCH-related diagnostics. The GARCH escalation starts with ARCH(1), then tries larger ARCH orders, then GARCH(1,1), then GARCH(2,1) and GARCH(1,2). It keeps the most parsimonious model that reduces residual ARCH-LM and squared-residual Ljung-Box diagnostics.

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

PRED_DIR = OUTPUT_DIR / "06_prediction"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Config: edit only this block
# -----------------------------
PREDICT_MODE = "fast"        # "fast" or "full"
RIDGE_ALPHA_FAST = 1.0
USE_RIDGECV_FULL = False
PREDICTION_STRIDE = 5        # 1 = every trading day; 5 = faster weekly-style evaluation
MAX_FEATURE_SETS = "core"    # "core", "cluster", or "all"
INCLUDE_RETURN_TARGETS = False

if PREDICT_MODE == "full":
    PREDICTION_STRIDE = 1
    USE_RIDGECV_FULL = True
    MAX_FEATURE_SETS = "all"


MARKET_PATHS = {
    "sp500": SP500_PATH,
    "dowjones": DJ_PATH,
    "nasdaq100": NASDAQ_PATH,
    "russell2000": RUSSELL_PATH,
}


def load_market_close(path: Path) -> pd.Series:
    if not path.exists():
        return pd.Series(dtype=float)

    df = pd.read_csv(path)
    date_col = next((c for c in df.columns if c.lower() in {"date", "time"}), df.columns[0])
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.set_index(date_col).sort_index()

    numeric_cols = df.select_dtypes("number").columns.tolist()
    if not numeric_cols:
        return pd.Series(dtype=float)

    normalized_names = {c.lower().replace(" ", "_"): c for c in df.columns}
    close_col = (
        normalized_names.get("adj_close")
        or normalized_names.get("adjusted_close")
        or normalized_names.get("close")
        or numeric_cols[0]
    )

    return pd.to_numeric(df[close_col], errors="coerce").dropna()


def build_market_targets(paths: dict[str, Path]) -> pd.DataFrame:
    out = []

    for name, path in paths.items():
        close = load_market_close(path)

        if close.empty:
            print(f"Missing/unusable market file: {name} -> {path}")
            continue

        ret = np.log(close).diff()

        df = pd.DataFrame(index=close.index)

        for h in TARGET_HORIZONS:
            # Forward realized volatility over next h trading days.
            df[f"{name}_rv_{h}d"] = np.sqrt(
                ret.pow(2).rolling(h).sum()
            ).shift(-h)

            if INCLUDE_RETURN_TARGETS:
                df[f"{name}_return_{h}d"] = (
                    ret.rolling(h).sum().shift(-h)
                )

        out.append(df)

    return pd.concat(out, axis=1).sort_index() if out else pd.DataFrame()


def make_prediction_features(indices: pd.DataFrame) -> pd.DataFrame:
    idx = indices.filter(like="_norm").copy()
    d_idx = idx.diff().add_suffix("_chg")
    lag_idx = idx.shift(1).add_suffix("_lag1")
    return pd.concat([idx, d_idx, lag_idx], axis=1)


def get_model() -> object:
    if USE_RIDGECV_FULL:
        return make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=RIDGE_ALPHAS),
        )

    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA_FAST),
    )


def rolling_ridge_oos_fast(
    X: pd.DataFrame,
    y: pd.Series,
    min_train: int = MIN_TRAIN,
    stride: int = PREDICTION_STRIDE,
) -> dict:
    # This join naturally drops weekends because market targets exist only on trading days.
    df = X.join(y.rename("target")).dropna()

    if len(df) < min_train + 60:
        return {
            "n_obs": len(df),
            "test_n": max(0, len(df) - min_train),
            "stride": stride,
            "model": "RidgeCV" if USE_RIDGECV_FULL else f"Ridge_alpha_{RIDGE_ALPHA_FAST}",
            "oos_r2": np.nan,
            "oos_rmse": np.nan,
            "oos_corr": np.nan,
        }

    xcols = X.columns.tolist()
    preds, actuals, baselines = [], [], []

    for t in range(min_train, len(df), stride):
        train = df.iloc[:t]
        test = df.iloc[[t]]

        model = get_model()
        model.fit(train[xcols], train["target"])

        preds.append(model.predict(test[xcols])[0])
        actuals.append(test["target"].iloc[0])
        baselines.append(train["target"].mean())

    pred = np.asarray(preds)
    actual = np.asarray(actuals)
    base = np.asarray(baselines)

    denom = np.sum((actual - base) ** 2)

    return {
        "n_obs": len(df),
        "test_n": len(actual),
        "stride": stride,
        "model": "RidgeCV" if USE_RIDGECV_FULL else f"Ridge_alpha_{RIDGE_ALPHA_FAST}",
        "oos_r2": 1 - np.sum((actual - pred) ** 2) / denom if denom > 0 else np.nan,
        "oos_rmse": float(np.sqrt(mean_squared_error(actual, pred))),
        "oos_corr": (
            float(np.corrcoef(actual, pred)[0, 1])
            if np.std(pred) > 0 and np.std(actual) > 0
            else np.nan
        ),
    }


def build_feature_sets(features: pd.DataFrame, labels: pd.Series) -> dict[str, pd.DataFrame]:
    base_sets = {
        "all_cluster_indices": features,
        "equal_indices_only": features.filter(like="_equal_norm"),
        "core_indices_only": features.filter(like="_core_equal_norm"),
        "stability_weighted_only": features.filter(like="_stability_weighted_norm"),
    }

    cluster_sets = {}

    for cl in sorted(labels.unique()):
        cl = int(cl)

        cluster_sets[f"cluster_{cl}_equal"] = features[
            [c for c in features.columns if c.startswith(f"cluster_{cl}_equal_norm")]
        ]

        cluster_sets[f"cluster_{cl}_core_equal"] = features[
            [c for c in features.columns if c.startswith(f"cluster_{cl}_core_equal_norm")]
        ]

        cluster_sets[f"cluster_{cl}_stability_weighted"] = features[
            [c for c in features.columns if c.startswith(f"cluster_{cl}_stability_weighted_norm")]
        ]

    if MAX_FEATURE_SETS == "core":
        feature_sets = {
            "equal_indices_only": base_sets["equal_indices_only"],
            "core_indices_only": base_sets["core_indices_only"],
            "stability_weighted_only": base_sets["stability_weighted_only"],
        }
    elif MAX_FEATURE_SETS in {"cluster", "all"}:
        feature_sets = {
            **base_sets,
            **cluster_sets,
        }
    else:
        raise ValueError("MAX_FEATURE_SETS must be 'core', 'cluster', or 'all'.")

    return {k: v for k, v in feature_sets.items() if v.shape[1] > 0}



targets = build_market_targets(MARKET_PATHS)
features = make_prediction_features(indices_norm)

features.to_csv(PRED_DIR / "prediction_features.csv")
targets.to_csv(PRED_DIR / "market_targets.csv")

ridge_rows = []

if not targets.empty:
    feature_sets = build_feature_sets(features, labels_final)

    print(f"Prediction mode: {PREDICT_MODE}")
    print(f"Stride: {PREDICTION_STRIDE}")
    print(f"Model: {'RidgeCV' if USE_RIDGECV_FULL else f'Ridge alpha={RIDGE_ALPHA_FAST}'}")
    print(f"Targets: {targets.shape[1]}")
    print(f"Feature sets: {len(feature_sets)}")

    for target in targets.columns:
        for feature_set, X in feature_sets.items():
            res = rolling_ridge_oos_fast(
                X,
                targets[target],
                min_train=MIN_TRAIN,
                stride=PREDICTION_STRIDE,
            )

            ridge_rows.append({
                "target": target,
                "feature_set": feature_set,
                "n_features": X.shape[1],
                **res,
            })

ridge_oos = pd.DataFrame(ridge_rows)
ridge_oos.to_csv(PRED_DIR / "rolling_ridge_oos_fast.csv", index=False)

ridge_oos.sort_values("oos_r2", ascending=False).head(20) if not ridge_oos.empty else ridge_oos

Prediction mode: fast
Stride: 5
Model: Ridge alpha=1.0
Targets: 8
Feature sets: 3


,target,feature_set,n_features,n_obs,test_n,stride,model,oos_r2,oos_rmse,oos_corr
17,nasdaq100_rv_5d,stability_weighted_only,18,1100,120,5,Ridge_alpha_1.0,0.188188,0.011815,0.365385
11,dowjones_rv_5d,stability_weighted_only,18,1100,120,5,Ridge_alpha_1.0,0.182419,0.014505,0.342534
15,nasdaq100_rv_5d,equal_indices_only,36,1100,120,5,Ridge_alpha_1.0,0.168060,0.011961,0.365811
9,dowjones_rv_5d,equal_indices_only,36,1100,120,5,Ridge_alpha_1.0,0.151755,0.014774,0.330750
5,sp500_rv_5d,stability_weighted_only,18,1100,120,5,Ridge_alpha_1.0,0.139165,0.009861,0.312346
16,nasdaq100_rv_5d,core_indices_only,18,1100,120,5,Ridge_alpha_1.0,0.107489,0.012388,0.322879
3,sp500_rv_5d,equal_indices_only,36,1100,120,5,Ridge_alpha_1.0,0.099628,0.010085,0.315956
10,dowjones_rv_5d,core_indices_only,18,1100,120,5,Ridge_alpha_1.0,0.097336,0.015241,0.297128
4,sp500_rv_5d,core_indices_only,18,1100,120,5,Ridge_alpha_1.0,0.072000,0.010239,0.253245
23,russell2000_rv_5d,stability_weighted_only,18,1100,120,5,Ridge_alpha_1.0,0.071382,0.012746,0.251754


In [89]:
# -----------------------------------------------------------------------------
# GARCH escalation diagnostics
# Requires daily return targets. RV targets remain for Ridge prediction.
# -----------------------------------------------------------------------------

from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

GARCH_DIR = OUTPUT_DIR / "06_prediction"
GARCH_DIR.mkdir(parents=True, exist_ok=True)

GARCH_RETURN_TARGETS = True
GARCH_EXOG_PATTERN = r"cluster_\d+_stability_weighted_norm_chg$"
GARCH_MIN_OBS = 250


def garch_diagnostics(std_resid: pd.Series) -> dict:
    z = pd.Series(std_resid).replace([np.inf, -np.inf], np.nan).dropna()

    if len(z) < 50:
        return {
            "arch_lm_pvalue": np.nan,
            "sq_ljungbox_pvalue": np.nan,
        }

    arch_p = het_arch(z, nlags=5)[1]
    lb_p = acorr_ljungbox(
        z.pow(2),
        lags=[10],
        return_df=True,
    )["lb_pvalue"].iloc[0]

    return {
        "arch_lm_pvalue": float(arch_p),
        "sq_ljungbox_pvalue": float(lb_p),
    }


def fit_garch_escalation(
    y: pd.Series,
    x: pd.DataFrame | None = None,
    min_obs: int = GARCH_MIN_OBS,
) -> pd.DataFrame:
    if not HAS_ARCH:
        return pd.DataFrame([{
            "note": "arch package not installed; run `pip install arch` to enable GARCH diagnostics."
        }])

    df = pd.DataFrame({"y": 100 * y}).join(x).replace([np.inf, -np.inf], np.nan).dropna()

    if len(df) < min_obs:
        return pd.DataFrame([{
            "note": "not enough observations for GARCH diagnostics",
            "n_obs": len(df),
        }])

    xcols = [c for c in df.columns if c != "y"]
    X = df[xcols] if xcols else None

    specs = [
        ("ARCH(1)", 1, 0),
        ("ARCH(2)", 2, 0),
        ("ARCH(3)", 3, 0),
        ("GARCH(1,1)", 1, 1),
        ("GARCH(2,1)", 2, 1),
        ("GARCH(1,2)", 1, 2),
    ]

    rows = []

    for name, p, q in specs:
        try:
            mod = arch_model(
                df["y"],
                x=X,
                mean="ARX" if X is not None else "Constant",
                lags=1,
                vol="GARCH",
                p=p,
                q=q,
                dist="normal",
                rescale=False,
            )

            res = mod.fit(disp="off", show_warning=False)
            diag = garch_diagnostics(res.std_resid)

            rows.append({
                "spec": name,
                "p": p,
                "q": q,
                "n_obs": len(df),
                "n_exog": 0 if X is None else X.shape[1],
                "aic": float(res.aic),
                "bic": float(res.bic),
                **diag,
                "passes_arch_lm_5pct": (
                    bool(diag["arch_lm_pvalue"] > 0.05)
                    if pd.notna(diag["arch_lm_pvalue"])
                    else False
                ),
                "passes_sq_ljungbox_5pct": (
                    bool(diag["sq_ljungbox_pvalue"] > 0.05)
                    if pd.notna(diag["sq_ljungbox_pvalue"])
                    else False
                ),
            })

        except Exception as e:
            rows.append({
                "spec": name,
                "p": p,
                "q": q,
                "error": str(e),
            })

    out = pd.DataFrame(rows)

    if {"passes_arch_lm_5pct", "passes_sq_ljungbox_5pct", "aic"}.issubset(out.columns):
        out["recommended"] = False

        ok = out[
            out["passes_arch_lm_5pct"]
            & out["passes_sq_ljungbox_5pct"]
            & out["aic"].notna()
        ].sort_values(["p", "q", "aic"])

        if len(ok):
            pick = ok.index[0]
        else:
            pick = out.loc[out["aic"].notna()].sort_values("aic").index[0]

        out.loc[pick, "recommended"] = True

    return out


# -----------------------------------------------------------------------------
# Build daily return targets directly from market close files
# -----------------------------------------------------------------------------

def build_garch_return_targets(paths: dict[str, Path]) -> pd.DataFrame:
    out = []

    for name, path in paths.items():
        close = load_market_close(path)

        if close.empty:
            print(f"Missing/unusable market file for GARCH: {name} -> {path}")
            continue

        ret = np.log(close).diff()
        out.append(pd.DataFrame({f"{name}_return_1d": ret}))

    return pd.concat(out, axis=1).sort_index() if out else pd.DataFrame()

In [90]:
garch_targets = build_garch_return_targets(MARKET_PATHS)

# Use only one compact exogenous set: stability-weighted cluster index changes.
x_garch = features.filter(regex=GARCH_EXOG_PATTERN)

garch_rows = []

if not garch_targets.empty:
    print(f"GARCH targets: {garch_targets.shape[1]}")
    print(f"GARCH exogenous variables: {x_garch.shape[1]}")

    for target in garch_targets.columns:
        diag = fit_garch_escalation(
            garch_targets[target],
            x=x_garch,
            min_obs=GARCH_MIN_OBS,
        )

        diag.insert(0, "target", target)
        garch_rows.append(diag)

garch_results = (
    pd.concat(garch_rows, ignore_index=True)
    if garch_rows
    else pd.DataFrame([{"note": "No market return targets available."}])
)

garch_results.to_csv(
    GARCH_DIR / "garch_escalation_diagnostics.csv",
    index=False,
)

garch_results.head(30)

GARCH targets: 4
GARCH exogenous variables: 6


,target,spec,p,q,n_obs,n_exog,aic,bic,arch_lm_pvalue,sq_ljungbox_pvalue,passes_arch_lm_5pct,passes_sq_ljungbox_5pct,recommended
0,sp500_return_1d,ARCH(1),1,0,1104,6,2975.178963,3025.236853,4.808887e-12,1.307112e-12,False,False,False
1,sp500_return_1d,ARCH(2),2,0,1104,6,2962.702287,3017.765966,3.674714e-12,2.587403e-12,False,False,False
2,sp500_return_1d,ARCH(3),3,0,1104,6,2900.981288,2961.050757,4.840769e-02,2.951027e-02,False,False,False
3,sp500_return_1d,"GARCH(1,1)",1,1,1104,6,2849.223236,2904.286915,2.138187e-02,4.141131e-02,False,False,False
4,sp500_return_1d,"GARCH(2,1)",2,1,1104,6,2851.223235,2911.292703,2.136607e-02,4.139337e-02,False,False,False
5,sp500_return_1d,"GARCH(1,2)",1,2,1104,6,2847.783645,2907.853113,6.223149e-02,1.049259e-01,True,True,True
6,dowjones_return_1d,ARCH(1),1,0,1104,6,3935.293077,3985.350968,2.414177e-13,2.028328e-18,False,False,False
7,dowjones_return_1d,ARCH(2),2,0,1104,6,3916.443024,3971.506703,2.643549e-12,9.091760e-12,False,False,False
8,dowjones_return_1d,ARCH(3),3,0,1104,6,3864.851364,3924.920832,3.713540e-04,8.800040e-05,False,False,False
9,dowjones_return_1d,"GARCH(1,1)",1,1,1104,6,3779.521651,3834.585330,1.056125e-01,4.462514e-01,True,True,True


## 7. Minimal validity checks

These checks are CSV-first: event concentration by cluster and a small label-permutation benchmark for the best prediction result.

In [91]:
def event_concentration(panel: pd.DataFrame, labels: pd.Series, q: float = 0.95) -> pd.DataFrame:
    rows = []
    for cl in sorted(labels.unique()):
        members = labels[labels == cl].index
        z = panel[members]
        high_counts = z.gt(z.quantile(q)).sum().sort_values(ascending=False)
        total = high_counts.sum()
        rows.append({
            "cluster": cl,
            "n_terms": len(members),
            "top_term": high_counts.index[0] if len(high_counts) else None,
            "top_term_event_share": float(high_counts.iloc[0] / total) if total else np.nan,
            "top_5_event_share": float(high_counts.head(5).sum() / total) if total else np.nan,
        })
    return pd.DataFrame(rows)

validity_event = event_concentration(panel_norm, labels_final)
validity_event.to_csv(OUTPUT_DIR / "07_validity_checks" / "event_concentration.csv", index=False)

run_manifest = pd.DataFrame([{
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "start_date": START_DATE,
    "end_date": END_DATE,
    "n_terms": panel_norm.shape[1],
    "n_days": panel_norm.shape[0],
    "sax_source": "tslearn.piecewise.SymbolicAggregateApproximation",
    "n_segments": N_SEGMENTS,
    "alphabet_size": ALPHABET_SIZE,
    "distance": "sax_mindist",
    "linkage": "complete",
    "final_k": FINAL_K,
    "stability_subsamples": STABILITY_SUBSAMPLES,
    "stability_fraction": STABILITY_FRACTION,
}])
run_manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)

print("Saved analysis outputs under:", OUTPUT_DIR)
print("Key files:")
for p in [
    OUTPUT_DIR / "02_sax" / "sax_strings.csv",
    OUTPUT_DIR / "03_hierarchical_clustering" / "cluster_assignments.csv",
    OUTPUT_DIR / "04_stability" / "cluster_selection_by_k.csv",
    OUTPUT_DIR / "05_cluster_indices" / "cluster_indices.csv",
    OUTPUT_DIR / "06_prediction" / "rolling_ridge_oos.csv",
    OUTPUT_DIR / "06_prediction" / "garch_escalation_diagnostics.csv",
    OUTPUT_DIR / "03_hierarchical_clustering" / "sax_mindist_matrix.csv",
]:
    print("-", p)

Saved analysis outputs under: C:\Python\CSUREMM\output\sax_tests_july_09
Key files:
- C:\Python\CSUREMM\output\sax_tests_july_09\02_sax\sax_strings.csv
- C:\Python\CSUREMM\output\sax_tests_july_09\03_hierarchical_clustering\cluster_assignments.csv
- C:\Python\CSUREMM\output\sax_tests_july_09\04_stability\cluster_selection_by_k.csv
- C:\Python\CSUREMM\output\sax_tests_july_09\05_cluster_indices\cluster_indices.csv
- C:\Python\CSUREMM\output\sax_tests_july_09\06_prediction\rolling_ridge_oos.csv
- C:\Python\CSUREMM\output\sax_tests_july_09\06_prediction\garch_escalation_diagnostics.csv
- C:\Python\CSUREMM\output\sax_tests_july_09\03_hierarchical_clustering\sax_mindist_matrix.csv


## 8. Poster Graphs

In [ ]:
# =============================================================================
# 8.1 Options for cleaner cluster-shape visualization
# Options:
#   1. SAX reconstruction
#   2. Cluster medoid / representative normalized series
#   3. Combined: medoid series + SAX reconstruction
# =============================================================================

POSTER_DIR = OUTPUT_DIR / "08_poster"
POSTER_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 8.1 SAX segment centroid profiles
# Full-window average symbolic SAX values, clean line only
# No auto-selected window, no shaded bands, no SD
# -----------------------------------------------------------------------------

CENTROID_DIR = OUTPUT_DIR / "08_poster" / "sax_centroid_profiles"
CENTROID_DIR.mkdir(parents=True, exist_ok=True)

# Uses:
#   sax_code_df     rows = terms, columns = seg_1 ... seg_96
#   labels_final    term -> cluster assignment
#   clusters        sorted cluster labels
#   N_SEGMENTS      96
#   OUTPUT_DIR

full_selected_cols = [f"seg_{i}" for i in range(1, N_SEGMENTS + 1)]

centroid_records = []

for cl in clusters:
    members = labels_final[labels_final == cl].index.intersection(sax_code_df.index)
    cluster_sax = sax_code_df.loc[members, full_selected_cols]

    centroid = cluster_sax.mean(axis=0)

    x = np.arange(1, N_SEGMENTS + 1)

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(
        x,
        centroid.values,
        marker="o",
        linewidth=2.6,
        markersize=4.5,
    )

    ax.set_title(
        f"Cluster {int(cl)}: SAX Segment Centroid Profile",
        fontsize=18,
        fontweight="bold",
        pad=14,
    )

    ax.set_xlabel("PAA segment position within SAX representation", fontsize=12)
    ax.set_ylabel("Average symbolic level", fontsize=12)

    ax.set_xticks(np.arange(1, N_SEGMENTS + 1, 5))
    ax.set_xlim(1, N_SEGMENTS)

    ymin = centroid.min()
    ymax = centroid.max()
    yrange = ymax - ymin

    if yrange == 0:
        ax.set_ylim(ymin - 0.5, ymax + 0.5)
    else:
        pad = 0.10 * yrange
        ax.set_ylim(ymin - pad, ymax + pad)

    ax.grid(alpha=0.25)

    ax.text(
        0.02,
        0.05,
        f"n = {len(members)} terms",
        transform=ax.transAxes,
        fontsize=10,
        color="gray",
    )

    plt.tight_layout()

    fig.savefig(
        CENTROID_DIR / f"cluster_{int(cl)}_sax_segment_centroid_profile_full.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    for pos, val in zip(x, centroid.values):
        centroid_records.append(
            {
                "cluster": int(cl),
                "sax_segment": int(pos),
                "mean_symbolic_level": float(val),
                "n_terms": int(len(members)),
            }
        )

centroid_profiles_full = pd.DataFrame(centroid_records)

centroid_profiles_full.to_csv(
    CENTROID_DIR / "cluster_sax_segment_centroid_profiles.csv",
    index=False,
)

centroid_profiles

,cluster,local_position,sax_segment,mean_sax_code,q25_sax_code,q75_sax_code,n_terms
0,1,1,16,4.424658,3.0,6.0,219
1,1,2,17,5.200913,4.0,7.0,219
2,1,3,18,4.200913,3.0,5.0,219
3,1,4,19,4.863014,4.0,6.0,219
4,1,5,20,6.031963,5.0,8.0,219
...,...,...,...,...,...,...,...
91,6,12,27,5.769231,4.0,8.0,39
92,6,13,28,4.743590,3.0,7.5,39
93,6,14,29,3.692308,2.0,4.0,39
94,6,15,30,4.205128,3.0,5.5,39


In [116]:
# -----------------------------------------------------------------------------
# 8.1 SAX centroid profiles: poster-ready 2x3 shared-scale panel
# -----------------------------------------------------------------------------

CENTROID_DIR = OUTPUT_DIR / "08_poster" / "sax_centroid_profiles_full"
CENTROID_DIR.mkdir(parents=True, exist_ok=True)

full_selected_cols = [f"seg_{i}" for i in range(1, N_SEGMENTS + 1)]

cluster_centroids = {}
cluster_ns = {}

for cl in clusters:
    members = labels_final[labels_final == cl].index.intersection(sax_code_df.index)
    cluster_sax = sax_code_df.loc[members, full_selected_cols]

    cluster_centroids[int(cl)] = cluster_sax.mean(axis=0).values
    cluster_ns[int(cl)] = len(members)

x = np.arange(1, N_SEGMENTS + 1)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(17, 8.8),
    sharex=True,
    sharey=True,
)

axes = axes.reshape(-1)

for ax, cl in zip(axes, sorted(cluster_centroids.keys())):
    y = cluster_centroids[int(cl)]

    ax.plot(
        x,
        y,
        linewidth=2.0,
        marker="o",
        markersize=3.0,
    )

    ax.set_title(
        f"Cluster {int(cl)}",
        fontsize=15,
        fontweight="bold",
        pad=8,
    )

    ax.text(
        0.03,
        0.08,
        f"n = {cluster_ns[int(cl)]} terms",
        transform=ax.transAxes,
        fontsize=10,
        color="gray",
    )

    ax.set_xticks(np.arange(1, N_SEGMENTS + 1, 10))
    ax.grid(alpha=0.22)

for ax in axes[::3]:
    ax.set_ylabel("Average symbolic level", fontsize=11)

for ax in axes[-3:]:
    ax.set_xlabel("PAA segment position", fontsize=11)

fig.suptitle(
    "SAX segment centroid profiles by attention cluster",
    fontsize=20,
    fontweight="bold",
    y=0.98,
)

fig.text(
    0.5,
    0.025,
    "Each line is the cluster-average SAX symbol value across the full 96-segment representation. "
    "Shared y-axis enables comparison across clusters.",
    ha="center",
    fontsize=10,
    color="gray",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.94])

fig.savefig(
    CENTROID_DIR / "fig_sax_centroid_profiles_shared_panel.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [119]:
# -----------------------------------------------------------------------------
# 8.1 SAX centroid profiles: compressed 16-block poster version
# Uses median within each block to respect the symbolic/discrete SAX scale
# -----------------------------------------------------------------------------

COMPRESSED_SEGMENTS = 16

COMPRESSED_DIR = OUTPUT_DIR / "08_poster" / "sax_centroid_profiles_compressed"
COMPRESSED_DIR.mkdir(parents=True, exist_ok=True)

full_sax_cols = [f"seg_{i}" for i in range(1, N_SEGMENTS + 1)]

# 96 SAX positions -> 16 displayed blocks
# each displayed block summarizes 6 adjacent SAX positions
segment_bins = np.array_split(np.arange(1, N_SEGMENTS + 1), COMPRESSED_SEGMENTS)

cluster_profiles = {}
cluster_ns = {}
compressed_records = []

for cl in sorted(clusters):
    members = labels_final[labels_final == cl].index.intersection(sax_code_df.index)
    cluster_sax = sax_code_df.loc[members, full_sax_cols]

    # cluster centroid at each of the original 96 SAX positions
    centroid_full = cluster_sax.mean(axis=0)

    compressed_profile = []

    for block_id, segs in enumerate(segment_bins, start=1):
        cols = [f"seg_{i}" for i in segs]

        # median is more appropriate than mean for symbolic SAX values
        block_value = centroid_full.loc[cols].median()

        compressed_profile.append(block_value)

        compressed_records.append(
            {
                "cluster": int(cl),
                "compressed_block": int(block_id),
                "original_segment_start": int(segs[0]),
                "original_segment_end": int(segs[-1]),
                "median_centroid_symbolic_level": float(block_value),
                "n_terms": int(len(members)),
            }
        )

    cluster_profiles[int(cl)] = np.array(compressed_profile)
    cluster_ns[int(cl)] = len(members)

compressed_profile_table = pd.DataFrame(compressed_records)

compressed_profile_table.to_csv(
    COMPRESSED_DIR / "cluster_sax_centroid_profiles_16_blocks.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Plot 2x3 shared-scale panel
# -----------------------------------------------------------------------------

x = np.arange(1, COMPRESSED_SEGMENTS + 1)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(16, 8.6),
    sharex=True,
    sharey=True,
)

axes = axes.reshape(-1)

for ax, cl in zip(axes, sorted(cluster_profiles.keys())):
    y = cluster_profiles[int(cl)]

    ax.plot(
        x,
        y,
        marker="o",
        linewidth=2.7,
        markersize=5.5,
    )

    ax.set_title(
        f"Cluster {int(cl)}",
        fontsize=15,
        fontweight="bold",
        pad=8,
    )

    ax.text(
        0.04,
        0.08,
        f"n = {cluster_ns[int(cl)]} terms",
        transform=ax.transAxes,
        fontsize=10,
        color="gray",
    )

    ax.set_xticks(x)
    ax.grid(alpha=0.22)

for ax in axes[::3]:
    ax.set_ylabel("Median symbolic level", fontsize=11)

for ax in axes[-3:]:
    ax.set_xlabel("Compressed SAX block", fontsize=11)

fig.suptitle(
    "Compressed SAX centroid shapes by attention cluster",
    fontsize=20,
    fontweight="bold",
    y=0.98,
)

fig.text(
    0.5,
    0.025,
    "Original 96-position SAX centroid profiles are compressed into 16 blocks. "
    "Each point is the median centroid symbolic level across 6 adjacent SAX positions.",
    ha="center",
    fontsize=10,
    color="gray",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.94])

fig.savefig(
    COMPRESSED_DIR / "fig_sax_centroid_profiles_16_blocks_shared_panel.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

compressed_profile_table

,cluster,compressed_block,original_segment_start,original_segment_end,median_centroid_symbolic_level,n_terms
0,1,1,1,6,4.130137,219
1,1,2,7,12,4.360731,219
2,1,3,13,18,4.392694,219
3,1,4,19,24,4.598174,219
4,1,5,25,30,4.920091,219
...,...,...,...,...,...,...
91,6,12,67,72,4.961538,39
92,6,13,73,78,4.679487,39
93,6,14,79,84,4.294872,39
94,6,15,85,90,4.730769,39


In [144]:
# -----------------------------------------------------------------------------
# 8.1k Shape-aligned time-series panel with dendrogram
# Similar to classic time-series clustering display:
#   left  = term labels
#   mid   = normalized term trajectories
#   right = dendrogram using existing cluster/SAX feature structure
# -----------------------------------------------------------------------------

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

POSTER_DIR = OUTPUT_DIR / "08_poster"
POSTER_DIR.mkdir(parents=True, exist_ok=True)

CLASSIC_DIR = POSTER_DIR / "classic_cluster_timeseries_panel"
CLASSIC_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------------

TOP_N_PER_CLUSTER = 2       # 6 clusters × 6 terms = 36 rows, like the reference
SMOOTH_WINDOW = 7           # visual smoothing only
FIG_HEIGHT_PER_ROW = 0.30


# -----------------------------------------------------------------------------
# Select representative terms from each cluster
# closest to each cluster center in SAX space
# -----------------------------------------------------------------------------

selected_terms = []
selected_records = []

full_sax_cols = [f"seg_{i}" for i in range(1, N_SEGMENTS + 1)]

for cl in sorted(labels_final.unique()):
    members = labels_final[labels_final == cl].index.intersection(sax_code_df.index)
    cluster_sax = sax_code_df.loc[members, full_sax_cols]

    center = cluster_sax.mean(axis=0)
    dist_to_center = np.sqrt(((cluster_sax - center) ** 2).sum(axis=1))

    reps = dist_to_center.sort_values().head(TOP_N_PER_CLUSTER).index.tolist()

    for rank, term in enumerate(reps, start=1):
        selected_terms.append(term)
        selected_records.append(
            {
                "cluster": int(cl),
                "rank_within_cluster": rank,
                "term": term,
                "distance_to_cluster_center": float(dist_to_center.loc[term]),
            }
        )

selected_terms = [t for t in selected_terms if t in panel_norm.columns]

selected_term_table = pd.DataFrame(selected_records)
selected_term_table.to_csv(
    CLASSIC_DIR / "classic_panel_selected_terms.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Build matrices
# -----------------------------------------------------------------------------

# only selecting 2025-2026

# ------------------------------------------------------------------
# Display only the 6-month period around the end of 2025
# ------------------------------------------------------------------

DISPLAY_START = "2025-01-01"
DISPLAY_END   = "2026-12-31"

panel_display = panel_norm.copy()
panel_display.index = pd.to_datetime(panel_display.index)

panel_display = panel_display.loc[
    DISPLAY_START:DISPLAY_END
]

# plot_panel = panel_norm[selected_terms].copy()
plot_panel = panel_display[selected_terms].copy()

# z-score each row again for visual comparability
plot_matrix = plot_panel.T
plot_matrix = plot_matrix.sub(plot_matrix.mean(axis=1), axis=0)
plot_matrix = plot_matrix.div(plot_matrix.std(axis=1).replace(0, np.nan), axis=0)
plot_matrix = plot_matrix.fillna(0)

# smooth only for display
plot_matrix_smooth = (
    plot_matrix
    .T
    .rolling(SMOOTH_WINDOW, center=True, min_periods=1)
    .mean()
    .T
)

# dendrogram feature matrix: use SAX codes for the selected terms
dendro_X = sax_code_df.loc[selected_terms, full_sax_cols].to_numpy(dtype=float)

Z = linkage(dendro_X, method="average", metric="euclidean")

dendro_info = dendrogram(
    Z,
    orientation="right",
    no_plot=True,
    labels=selected_terms,
)

ordered_terms = dendro_info["ivl"]

plot_matrix_smooth = plot_matrix_smooth.loc[ordered_terms]

term_to_cluster = labels_final.astype(int).to_dict()


# -----------------------------------------------------------------------------
# Plot classic panel
# -----------------------------------------------------------------------------

n_rows = len(ordered_terms)

FIG_HEIGHT_PER_ROW = 0.75
lane_height = 4.0
linewidth = 2.6

# Match scipy dendrogram's native leaf coordinates: 5, 15, 25, ...
y_positions = np.arange(n_rows) * 10 + 5
ylim = (0, n_rows * 10)

fig_height = max(9, FIG_HEIGHT_PER_ROW * n_rows)

fig = plt.figure(figsize=(13, fig_height))

gs = fig.add_gridspec(
    nrows=1,
    ncols=3,
    width_ratios=[2.0, 12.0, 2.0],
    wspace=0.03,
)

ax_labels = fig.add_subplot(gs[0, 0])
ax_ts = fig.add_subplot(gs[0, 1])
ax_den = fig.add_subplot(gs[0, 2])


# -----------------------------------------------------------------------------
# Left labels
# -----------------------------------------------------------------------------

ax_labels.set_xlim(0, 1)
ax_labels.set_ylim(*ylim)
ax_labels.axis("off")

for i, term in enumerate(ordered_terms):
    cl = term_to_cluster.get(term, None)
    ax_labels.text(
        0.98,
        y_positions[i],
        f"C{cl}: {term}",
        ha="right",
        va="center",
        fontsize=11,
        fontweight="bold",
    )


# -----------------------------------------------------------------------------
# Middle time-series traces
# -----------------------------------------------------------------------------

dates = pd.to_datetime(plot_matrix_smooth.columns)
x = np.arange(len(dates))

for i, term in enumerate(ordered_terms):
    y = plot_matrix_smooth.loc[term].to_numpy(dtype=float)

    max_abs = np.nanmax(np.abs(y))
    if not np.isfinite(max_abs) or max_abs == 0:
        y_scaled = np.zeros_like(y)
    else:
        y_scaled = y / max_abs * lane_height

    ax_ts.plot(
        x,
        y_positions[i] + y_scaled,
        linewidth=linewidth,
    )

ax_ts.set_ylim(*ylim)
ax_ts.set_yticks([])
ax_ts.set_xlim(-5, len(x) + 5)

month_starts = pd.date_range(
    dates.min().normalize(),
    dates.max().normalize(),
    freq="3MS",
)

tick_positions = [
    np.argmin(np.abs((dates - d).days))
    for d in month_starts
]

ax_ts.set_xticks(tick_positions)
ax_ts.set_xticklabels(
    [d.strftime("%b\n%Y") for d in month_starts],
    fontsize=9,
)

ax_ts.set_xlabel("Date")
ax_ts.grid(axis="x", alpha=0.18)

for spine in ["left", "right", "top"]:
    ax_ts.spines[spine].set_visible(False)


# -----------------------------------------------------------------------------
# Right dendrogram
# -----------------------------------------------------------------------------

dendrogram(
    Z,
    orientation="right",
    labels=selected_terms,
    ax=ax_den,
    color_threshold=None,
    above_threshold_color="black",
    link_color_func=lambda k: "black",
)

ax_den.set_ylim(*ylim)
ax_den.set_yticklabels([])
ax_den.set_xticks([])

for coll in ax_den.collections:
    coll.set_linewidth(2.0)

for spine in ["left", "right", "top", "bottom"]:
    ax_den.spines[spine].set_visible(False)

# -----------------------------------------------------------------------------
# Final formatting
# -----------------------------------------------------------------------------

# fig.suptitle(
#     "Representative search trajectories by SAX–MINDIST cluster",
#     fontsize=17,
#     fontweight="bold",
#     y=0.995,
# )

# fig.text(
#     0.5,
#     0.01,
#     f"Shown terms are the {TOP_N_PER_CLUSTER} closest to each cluster center in SAX space. "
#     "Traces are normalized and rescaled within row for shape comparison; dendrogram uses SAX-code distances.",
#     ha="center",
#     fontsize=9,
#     color="gray",
# )

plt.tight_layout(rect=[0, 0.03, 1, 0.97])

fig.savefig(
    CLASSIC_DIR / f"fig_classic_timeseries_dendrogram_panel_{TOP_N_PER_CLUSTER}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [142]:
plt.tight_layout(rect=[0, 0.03, 1, 0.97])

fig.savefig(
    CLASSIC_DIR / f"fig_classic_timeseries_dendrogram_panel_{TOP_N_PER_CLUSTER}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [141]:
# -----------------------------------------------------------------------------
# Plot classic panel
# -----------------------------------------------------------------------------

n_rows = len(ordered_terms)

FIG_HEIGHT_PER_ROW = 0.75
lane_gap = 1.35
lane_height = 1.05
linewidth = 2.6

y_positions = (n_rows - 1 - np.arange(n_rows)) * lane_gap
ylim = (-0.5, y_positions.max() + 0.5)

fig_height = max(9, FIG_HEIGHT_PER_ROW * n_rows)

fig = plt.figure(figsize=(13, fig_height))

gs = fig.add_gridspec(
    nrows=1,
    ncols=3,
    width_ratios=[2.0, 12.0, 2.0],
    wspace=0.03,
)

ax_labels = fig.add_subplot(gs[0, 0])
ax_ts = fig.add_subplot(gs[0, 1])
ax_den = fig.add_subplot(gs[0, 2])


# -----------------------------------------------------------------------------
# Left labels
# -----------------------------------------------------------------------------

ax_labels.set_xlim(0, 1)
ax_labels.set_ylim(*ylim)
ax_labels.axis("off")

for i, term in enumerate(ordered_terms):
    cl = term_to_cluster.get(term, None)
    ax_labels.text(
        0.98,
        y_positions[i],
        f"C{cl}: {term}",
        ha="right",
        va="center",
        fontsize=11,
        fontweight="bold",
    )


# -----------------------------------------------------------------------------
# Middle time-series traces
# -----------------------------------------------------------------------------

dates = plot_matrix_smooth.columns
x = np.arange(len(dates))

for i, term in enumerate(ordered_terms):
    y = plot_matrix_smooth.loc[term].to_numpy(dtype=float)

    max_abs = np.nanmax(np.abs(y))
    if not np.isfinite(max_abs) or max_abs == 0:
        y_scaled = np.zeros_like(y)
    else:
        y_scaled = y / max_abs * lane_height / 2

    ax_ts.plot(
        x,
        y_positions[i] + y_scaled,
        linewidth=linewidth,
    )

ax_ts.set_ylim(*ylim)
ax_ts.set_yticks([])
ax_ts.set_xlim(-5, len(x) + 5)

dates = pd.to_datetime(plot_matrix_smooth.columns)

month_starts = pd.date_range(
    dates.min().normalize(),
    dates.max().normalize(),
    freq="3MS",
)

tick_positions = [
    np.argmin(np.abs((dates - d).days))
    for d in month_starts
]

ax_ts.set_xticks(tick_positions)
ax_ts.set_xticklabels(
    [d.strftime("%b\n%Y") for d in month_starts],
    fontsize=9,
)

ax_ts.set_xlabel("Date")
ax_ts.grid(axis="x", alpha=0.18)

for spine in ["left", "right", "top"]:
    ax_ts.spines[spine].set_visible(False)


# -----------------------------------------------------------------------------
# Right dendrogram
# -----------------------------------------------------------------------------

dendrogram(
    Z,
    orientation="right",
    labels=selected_terms,
    ax=ax_den,
    color_threshold=None,
    above_threshold_color="black",
    link_color_func=lambda k: "black",
)

ax_den.set_ylim(*ylim)
ax_den.set_yticklabels([])
ax_den.set_xticks([])

for coll in ax_den.collections:
    coll.set_linewidth(2.0)

for spine in ["left", "right", "top", "bottom"]:
    ax_den.spines[spine].set_visible(False)

In [ ]:
# -----------------------------------------------------------------------------
# 8.2 Cluster summary table for poster
# -----------------------------------------------------------------------------

poster_cluster_summary = cluster_summary.copy()

poster_cluster_summary = poster_cluster_summary.rename(
    columns={
        "cluster": "Cluster",
        "n_terms": "Terms",
        "median_term_stability": "Median stability",
        "mean_intra_mindist": "Mean intra-MINDIST",
        "mean_between_mindist": "Mean between-MINDIST",
        "mean_mindist_margin": "Mean MINDIST margin",
        "top_10_core_terms": "Representative core terms",
    }
)

poster_cluster_summary["Cluster"] = (
    "Cluster " + poster_cluster_summary["Cluster"].astype(int).astype(str)
)

for col in [
    "Median stability",
    "Mean intra-MINDIST",
    "Mean between-MINDIST",
    "Mean MINDIST margin",
]:
    poster_cluster_summary[col] = poster_cluster_summary[col].round(3)

poster_cluster_summary.to_csv(
    POSTER_DIR / "table_cluster_summary_poster.csv",
    index=False,
)

poster_cluster_summary


# -----------------------------------------------------------------------------
# 8.3 Forecasting performance by cluster feature set
# Uses ridge_oos from Section 6.
# This only summarizes feature sets that already exist in the notebook output.
# -----------------------------------------------------------------------------

ridge_plot = ridge_oos.copy()

if ridge_plot.empty:
    print("ridge_oos is empty; no prediction performance figures created.")

else:
    ridge_plot["target_index"] = ridge_plot["target"].str.replace(
        r"_(rv|return)_\d+d$",
        "",
        regex=True,
    )

    ridge_plot["horizon_days"] = (
        ridge_plot["target"]
        .str.extract(r"_(?:rv|return)_(\d+)d$")[0]
        .astype(float)
    )

    perf_table = (
        ridge_plot
        .groupby(["feature_set", "target_index", "horizon_days"])
        .agg(
            n_features=("n_features", "first"),
            n_obs=("n_obs", "mean"),
            test_n=("test_n", "mean"),
            oos_r2=("oos_r2", "mean"),
            oos_rmse=("oos_rmse", "mean"),
            oos_corr=("oos_corr", "mean"),
        )
        .reset_index()
        .sort_values(["horizon_days", "target_index", "oos_r2"], ascending=[True, True, False])
    )

    perf_table.to_csv(
        POSTER_DIR / "table_feature_set_performance_long.csv",
        index=False,
    )

    display(perf_table)


# -----------------------------------------------------------------------------
# 8.4 Heatmap: rows = feature sets, columns = market index
# Separate figure for each horizon.
# -----------------------------------------------------------------------------

if not ridge_oos.empty:
    for h in sorted(ridge_plot["horizon_days"].dropna().unique()):
        h_df = ridge_plot.loc[ridge_plot["horizon_days"] == h].copy()

        heatmap = (
            h_df
            .pivot_table(
                index="feature_set",
                columns="target_index",
                values="oos_r2",
                aggfunc="mean",
            )
            .sort_index()
        )

        fig, ax = plt.subplots(figsize=(10, 6))

        im = ax.imshow(heatmap.values, aspect="auto")

        ax.set_xticks(np.arange(heatmap.shape[1]))
        ax.set_yticks(np.arange(heatmap.shape[0]))

        ax.set_xticklabels(heatmap.columns, rotation=30, ha="right")
        ax.set_yticklabels(heatmap.index)

        ax.set_title(
            f"Out-of-sample $R^2$ by feature set and market index, {int(h)}-day horizon",
            fontsize=14,
            fontweight="bold",
            pad=14,
        )

        ax.set_xlabel("Market index")
        ax.set_ylabel("Feature set")

        for i in range(heatmap.shape[0]):
            for j in range(heatmap.shape[1]):
                value = heatmap.iloc[i, j]
                if pd.notna(value):
                    ax.text(
                        j,
                        i,
                        f"{value:.2f}",
                        ha="center",
                        va="center",
                        fontsize=9,
                    )

        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Mean OOS $R^2$")

        plt.tight_layout()

        fig.savefig(
            POSTER_DIR / f"fig_oos_r2_heatmap_{int(h)}d.png",
            dpi=300,
            bbox_inches="tight",
        )

        plt.show()


# -----------------------------------------------------------------------------
# 8.5 Poster-ready performance table
# Rows = feature sets, columns = metrics
# -----------------------------------------------------------------------------

if not ridge_oos.empty:
    poster_perf = (
        ridge_plot
        .groupby("feature_set")
        .agg(
            mean_oos_r2=("oos_r2", "mean"),
            best_oos_r2=("oos_r2", "max"),
            mean_oos_corr=("oos_corr", "mean"),
            best_oos_corr=("oos_corr", "max"),
            mean_oos_rmse=("oos_rmse", "mean"),
            mean_test_n=("test_n", "mean"),
            n_models=("oos_r2", "count"),
        )
        .reset_index()
        .sort_values("best_oos_r2", ascending=False)
    )

    poster_perf = poster_perf.rename(
        columns={
            "feature_set": "Feature set",
            "mean_oos_r2": "Mean OOS R²",
            "best_oos_r2": "Best OOS R²",
            "mean_oos_corr": "Mean OOS Corr.",
            "best_oos_corr": "Best OOS Corr.",
            "mean_oos_rmse": "Mean OOS RMSE",
            "mean_test_n": "Mean test n",
            "n_models": "Models",
        }
    )

    for col in [
        "Mean OOS R²",
        "Best OOS R²",
        "Mean OOS Corr.",
        "Best OOS Corr.",
        "Mean OOS RMSE",
    ]:
        poster_perf[col] = poster_perf[col].round(3)

    poster_perf["Mean test n"] = poster_perf["Mean test n"].round(0).astype(int)

    poster_perf.to_csv(
        POSTER_DIR / "table_prediction_performance_poster.csv",
        index=False,
    )

    poster_perf